# GrandNexus v7.1.0-rc — mesure sur un jeu FOLIO jamais vu (split `train`)
Moteur ref-2.2 (questions composées par réfutation), prompt v5.4. La validation est désormais vue en partie ; le pool neuf est le split d'entraînement (1 001 items), tiré avec la graine 1. Exécute dans l'ordre, sans adapter les cellules.

## 1. Récupérer GrandNexus

In [ ]:
import os, glob, zipfile, subprocess
DOSSIER_DRIVE = '/content/drive/MyDrive/GrandNexus'
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive non monté :', e)
zips = sorted(glob.glob(os.path.join(DOSSIER_DRIVE, 'GrandNexus_*.zip')) + glob.glob('/content/drive/MyDrive/GrandNexus_Sessions/GrandNexus_v7*.zip'), key=os.path.getmtime)
if zips:
    source = zips[-1]
else:
    from google.colab import files
    source = '/content/' + next(iter(files.upload()))
WORK = '/content/GrandNexus'
if not os.path.exists(WORK):
    zipfile.ZipFile(source).extractall('/content')
    print('décompressé depuis :', source)
print(subprocess.run('git describe --tags && git log --oneline -1', shell=True, cwd=WORK, capture_output=True, text=True).stdout)

## 2. Paquets, clé, contrôle avant vol (1 appel)

In [ ]:
!pip install -q anthropic datasets clingo sympy networkx psutil
import os, anthropic
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
m = anthropic.Anthropic().messages.create(model='claude-sonnet-5-5', max_tokens=10, messages=[{'role': 'user', 'content': 'Réponds seulement : OK'}])
print('réponse :', m.content[0].text, '| id :', m.id, '| modèle :', m.model)

## 3. Tests hors ligne (0 appel)

In [ ]:
%cd /content/GrandNexus
!python -W ignore tests_lab/test_nexus_logic.py | tail -1
!python -W ignore tools/diff_engines.py tests_lab/programmes_nx | tail -1
!python -W ignore tools/lab_reasoning.py 2>/dev/null | grep C1

## 4. 100 items du split `train`, jamais vus (graine 1) — environ 220 appels
Attendu d'après les programmes réels d'hier rejoués hors ligne : strict autour de 72-77 %, 0 à 1 faux « oui », quelques « contradictoire » à tort en moins ; Claude seul vers 80-88 %.

In [ ]:
!python tools/bench_folio.py --split train --seed 1 --n 100 --modes translator,direct --strict --save-programs

## 5. Analyse à l'aveugle (0 appel) : disjonctions, formules composées, différentiel réel

In [ ]:
import glob, os, re, collections
d = sorted(glob.glob('/content/GrandNexus/tests_lab/resultats/programmes/folio_train_*'), key=os.path.getmtime)[-1]
progs = [open(f, encoding='utf-8').read() for f in glob.glob(d + '/*.nx')]
q = [l for p in progs for l in p.splitlines() if l.startswith('?')]
print(os.path.basename(d), '|', len(progs), 'programmes | avec disjonction :', sum('|' in p for p in progs),
      '| questions composées :', sum(('&' in x or '|' in x or '(' in x.split('?')[1].strip()[:1]) and '=>' not in x for x in q),
      '| universelles :', sum('=>' in x for x in q), '| cannot express :', sum(p.count('# cannot express') for p in progs))
print(subprocess.run(['python', 'tools/diff_engines.py', d], cwd='/content/GrandNexus', capture_output=True, text=True).stdout.strip().splitlines()[-1])

## 6. Sauvegarder dans le Drive

In [ ]:
import shutil, time
for p in glob.glob('/content/GrandNexus/**/__pycache__', recursive=True): shutil.rmtree(p, ignore_errors=True)
subprocess.run('git add -A && git -c user.name=Khaled -c user.email=khaled@grandnexus.local commit -qm "v7.1.0-rc run"', shell=True, cwd='/content/GrandNexus')
nom = os.path.join(DOSSIER_DRIVE, 'GrandNexus_session_' + time.strftime('%Y%m%d_%H%M'))
os.makedirs(DOSSIER_DRIVE, exist_ok=True); shutil.make_archive(nom, 'zip', '/content', 'GrandNexus'); print('sauvegardé :', nom + '.zip')